# 1. Data Understanding - Customer Retention Analytics

## 📊 Business Context

Customer churn is a critical challenge in the telecommunications industry. When customers leave for competitors, companies lose:
- **Revenue**: Monthly recurring revenue from the customer
- **Acquisition costs**: Marketing and sales expenses already invested
- **Lifetime value**: Potential future revenue from long-term relationships

**Business Objective**: Build a predictive model to identify customers at risk of churning, enabling proactive retention strategies.

**Success Metrics**: 
- Accurately identify 70%+ of potential churners
- Minimize false positives to avoid unnecessary retention costs
- Provide actionable insights for retention campaigns

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
%matplotlib inline

In [2]:
# Load the dataset from Excel file
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
print(f"Dataset shape (raw): {df.shape}")

Dataset shape (raw): (7043, 33)


In [3]:
# Standardize column names to match expected format
df = df.rename(columns={
    'Gender': 'gender',
    'Senior Citizen': 'SeniorCitizen',
    'Tenure Months': 'tenure',
    'Phone Service': 'PhoneService',
    'Multiple Lines': 'MultipleLines',
    'Internet Service': 'InternetService',
    'Online Security': 'OnlineSecurity',
    'Online Backup': 'OnlineBackup',
    'Device Protection': 'DeviceProtection',
    'Tech Support': 'TechSupport',
    'Streaming TV': 'StreamingTV',
    'Streaming Movies': 'StreamingMovies',
    'Paperless Billing': 'PaperlessBilling',
    'Payment Method': 'PaymentMethod',
    'Monthly Charges': 'MonthlyCharges',
    'Total Charges': 'TotalCharges',
    'Churn Label': 'Churn'
})

# Drop unnecessary columns for modeling
columns_to_drop = ['Count', 'Country', 'State', 'City', 'Zip Code', 
                   'Lat Long', 'Latitude', 'Longitude', 'CustomerID',
                   'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')

print(f"Dataset shape (after cleaning): {df.shape}")
print(f"Columns: {len(df.columns)}")
df.head()

Dataset shape (after cleaning): (7043, 20)
Columns: 20


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes


In [4]:
# Display basic information about the dataset
print("Dataset Info:")
print(df.info())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   object 
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-n

In [5]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [6]:
# Display statistical summary
print("Statistical Summary:")
df.describe()

Statistical Summary:


,tenure,MonthlyCharges
count,7043.000000,7043.000000
mean,32.371149,64.761692
std,24.559481,30.090047
min,0.000000,18.250000
25%,9.000000,35.500000
50%,29.000000,70.350000
75%,55.000000,89.850000
max,72.000000,118.750000


In [7]:
# Check data types and categorical features
print("Numerical Features:")
print(df.select_dtypes(include=[np.number]).columns.tolist())
print("\nCategorical Features:")
print(df.select_dtypes(include=['object']).columns.tolist())

Numerical Features:
['tenure', 'MonthlyCharges']

Categorical Features:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']


In [8]:
# Check target variable distribution
if 'Churn' in df.columns:
    print("Churn Distribution:")
    print(df['Churn'].value_counts())
    print(f"\nChurn Rate: {df['Churn'].value_counts(normalize=True).iloc[1]:.2%}")

Churn Distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn Rate: 26.54%
